# Procrustes baseline vs trained adapters

Closed-form alignment (centering + orthogonal rotation + isotropic scale, fit on 2,000 prompts
from the adapters' own training corpus) swept across α exactly like the trained adapters, then
compared pair by pair.

The notebook runs in three independent parts:

| part | needs | does |
|---|---|---|
| **1. Parity** | CPU | shows that the Procrustes path and the adapter path see the same eval activations |
| **2. Recompute** *(optional)* | GPU | reruns Procrustes for chosen pairs through `shared_truth` and checks it against the saved results |
| **3. Analysis** | CPU | rebuilds the cross-pair table and aggregates from saved results, and checks them against `procrustes_n21_summary.json` |

**Population.** The pairs are pinned to the `pair_id`s recorded in the published
`procrustes_n21_summary.json`, never a directory glob. The sweep folder now also holds the
multi-seed, baseline, and full-test-set runs, and those use a different eval set.

In [1]:
# --- setup ------------------------------------------------------------------
import os, sys, subprocess
from pathlib import Path

BRANCH   = "rebuttal"          # switch to "main" once merged
REPO_DIR = Path("/content/shared-truth")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/KingTechnician/shared-truth.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
sys.path.insert(0, str(REPO_DIR))

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass

# Pin scikit-learn to the version the probes were saved with, prior to importing
from shared_truth import env
env.pin_sklearn()

import json, numpy as np
from shared_truth import storage, metrics, naming
print("shared_truth @", storage.provenance()["shared_truth_sha"])

model.skops:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

installing scikit-learn==1.8.0 skops (probes were saved with scikit-learn 1.8.0; runtime had 1.6.1)
scikit-learn 1.8.0 (matches probes)
shared_truth @ b35747aad2cf1db0ae75e2a1054907c524ed6960


In [2]:
# --- config -----------------------------------------------------------------
TIE_PP      = 0.5         # |Δ| within this counts as a tie (as published)
MAX_LEN     = 512         # Procrustes-path truncation length
N_TRAIN     = 2000        # Procrustes fit corpus size (as published)
TRAIN_SEED  = 42

RECOMPUTE     = True     # part 2 needs a GPU runtime
RECOMPUTE_IDS = [         # pairs to rerun; cheapest published pair by default
    "qwen2_5-1_5b-instruct_l17_to_llama3_2-3b-instruct_l12__truth",
]

REFERENCE_SUMMARY = "procrustes_n21_summary.json"
OUT_DIR = Path("/content/procrustes_rerun")

In [3]:
# --- data root + pinned population -----------------------------------------
ROOT = Path(storage.fetch_results())
SWEEPS = ROOT / storage.RESULTS_PREFIX
reference = json.load(open(ROOT / REFERENCE_SUMMARY))
POPULATION = sorted(r["pair_id"] for r in reference["rows"])
print(f"published Procrustes population: {len(POPULATION)} pairs (from {REFERENCE_SUMMARY})")

missing = [p for p in POPULATION if not (SWEEPS / p / "procrustes_sweep.json").exists()]
if missing:
    raise SystemExit("missing procrustes_sweep.json:\n  " + "\n  ".join(missing))

saved = {p: json.load(open(SWEEPS / p / "procrustes_sweep.json")) for p in POPULATION}
for p in POPULATION:
    print(f"  {naming.parse_sweep_dir(p)['variant']:<16} {saved[p]['display_label']}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 78 files:   0%|          | 0/78 [00:00<?, ?it/s]

published Procrustes population: 22 pairs (from procrustes_n21_summary.json)
  truth            Gemma-2-2B (L13) → Llama-3.2-3B (L12)
  truth            Gemma-2-2B (L13) → Qwen2.5-1.5B (L17)
  transfer_truth   Llama-3-8B (L12) → Gemma-2-2B (L13)
  truth            Llama-3-8B (L12) → Mistral-7B (L13)
  transfer_rsa     Llama-3-8B (L14) → Gemma-2-2B (L14)
  transfer_rsa     Llama-3-8B (L14) → Mistral-7B (L14)
  truth            Llama-3.2-3B (L12) → Gemma-2-2B (L13)
  transfer_truth   Llama-3.2-3B (L12) → Mistral-7B (L13)
  transfer_rsa     Llama-3.2-3B (L12) → Mistral-7B (L14)
  transfer_rsa     Llama-3.2-3B (L12) → Qwen2.5-1.5B (L12)
  transfer_truth   Llama-3.2-3B (L12) → Qwen2.5-1.5B (L12)
  truth            Llama-3.2-3B (L12) → Qwen2.5-1.5B (L17)
  transfer_rsa     Llama-3.2-3B (L13) → Gemma-2-2B (L14)
  truth            Mistral-7B (L13) → Gemma-7B (L19)
  transfer_rsa     Mistral-7B (L14) → Gemma-7B (L15)
  transfer_rsa     Qwen2.5-1.5B (L12) → Gemma-2-2B (L11)
  transfer_rsa     Qw

## Part 1 — extraction parity (CPU)

The two paths tokenize differently: the adapter sweep uses no truncation, and the Procrustes path
truncates at 512 tokens. On the **eval** statements that difference can only matter if some
statement is longer than 512 tokens for some model, so checking token lengths settles it without
a GPU.

The **train** corpus is also truncated, but that is part of how the Procrustes fit is defined, and
the adapters were never trained through this code. It does not affect whether the eval-time
comparison is like-for-like.

A second, independent check was saved at run time: `ceiling_drift_pp` compares the target probe's
accuracy on the Procrustes path's target activations against the adapter sweep's recorded
ceiling. A drift of exactly zero means the two paths agree on every target-side prediction.

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer

eval_df = load_dataset("KingTechnician/tiu-splits")["test"].to_pandas()
eval_texts = eval_df[eval_df["statement_type"] == "simple"]["text"].tolist()
print(f"eval set: {len(eval_texts)} simple statements\n")

models = sorted({saved[p]["source_model"] for p in POPULATION} |
                {saved[p]["target_model"] for p in POPULATION})
parity_ok = True
for m in models:
    tok = AutoTokenizer.from_pretrained(m)
    lens = [len(tok(t)["input_ids"]) for t in eval_texts]
    ok = max(lens) <= MAX_LEN
    parity_ok &= ok
    print(f"{'ok ' if ok else 'OVER'}  max {max(lens):>4} tokens  (p99 {int(np.percentile(lens, 99)):>3})  {m}")

drifts = {p: saved[p].get("ceiling_drift_pp", float("nan")) for p in POPULATION}
nonzero = {p: d for p, d in drifts.items() if abs(d) > 1e-9}
print(f"\nsaved ceiling drift: {len(POPULATION) - len(nonzero)}/{len(POPULATION)} pairs exactly 0.00 pp")
for p, d in sorted(nonzero.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {d:+.2f} pp  {p}")

print("\n" + ("✓ truncation is a no-op on every eval statement: the eval-time comparison is like-for-like"
              if parity_ok else
              "✗ some eval statements exceed MAX_LEN: the two paths see different activations for those items"))

README.md:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  739kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  164kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

ok   max   42 tokens  (p99  23)  mistralai/Mistral-7B-Instruct-v0.3

saved ceiling drift: 22/22 pairs exactly 0.00 pp

✓ truncation is a no-op on every eval statement: the eval-time comparison is like-for-like


## Part 2 — recompute (GPU, optional)

Reruns the pipeline for `RECOMPUTE_IDS` through `shared_truth` and compares each pair against its
saved `procrustes_sweep.json`. Each model is loaded once per pair: train and eval texts are
extracted in one pass and then split. That gives identical activations to the original, which ran
two passes, because each text is processed independently.

Exact equality isn't guaranteed: fp16 matrix multiplies can differ by the last bit across GPU
types. So the check is stated in items, not floating-point tolerance. One eval item is
100/1599 ≈ 0.06 pp.

New files use the full sweep row schema, including per-item `true_scores` and `false_scores`, so
**strict overshoot becomes computable for Procrustes**. The saved originals only kept accuracy
and AUROC.

In [ ]:
recomputed = {}
if not RECOMPUTE:
    print("RECOMPUTE = False — skipping (set it and switch to a GPU runtime to run)")
else:
    import torch, gc
    from datetime import datetime, timezone
    FORK = Path("/content/Closing-Backdoors-Via-Representation-Transfer")
    if not FORK.exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/withmartian/Closing-Backdoors-Via-Representation-Transfer.git",
                        str(FORK)], check=True)
    sys.path.append(str(FORK))
    from shared_truth import activations, probes, sweep

    activations.install_monkeypatch()
    train_ds = load_dataset("withmartian/code_backdoors_dev_prod_hh_rlhf_50percent")["train"]
    train_texts = [ex["prompt"] for ex in train_ds.shuffle(seed=TRAIN_SEED).select(range(N_TRAIN))]
    eval_records, _, _ = sweep.load_test_set(statement_type="simple")
    eval_labels = np.array([int(r["label"]) for r in eval_records])
    assert [r["text"] for r in eval_records] == eval_texts, "eval order differs from Part 1"
    n_tr = len(train_texts)

    for pid in RECOMPUTE_IDS:
        s = saved[pid]
        src, sl, tgt, tl = s["source_model"], s["source_layer"], s["target_model"], s["target_layer"]
        print(f"\n=== {s['display_label']} ===")
        probe, dirs = probes.load_probe(naming.probe_repo_for(tgt, tl))

        src_all = activations.extract_last_token(src, sl, train_texts + eval_texts, max_len=MAX_LEN)
        tgt_all = activations.extract_last_token(tgt, tl, train_texts + eval_texts, max_len=MAX_LEN)
        src_tr, src_ev = src_all[:n_tr], src_all[n_tr:]
        tgt_tr, tgt_ev = tgt_all[:n_tr], tgt_all[n_tr:]
        if np.isnan(src_all).any() or np.isnan(tgt_all).any():
            print("  ⚠ NaN in activations")

        proc = sweep.fit_procrustes(src_tr, tgt_tr)
        preds0, probs0 = probes.score_batch(probe, dirs, tgt_ev)
        base_acc = float((preds0 == eval_labels).mean())
        drift = (base_acc - s["ceiling"]["accuracy"]) * 100
        rows = sweep.alpha_sweep_with_adapter(proc, src_ev, tgt_ev, probe, dirs, eval_labels)

        n = len(eval_labels)
        d_items = [round(abs(a["accuracy"] - b["accuracy"]) * n) for a, b in zip(rows, s["procrustes_sweep"])]
        d_auc = max(abs(a["auroc"] - b["auroc"]) for a, b in zip(rows, s["procrustes_sweep"]))
        print(f"  scale {proc.scale:.6f} (saved {s['procrustes_scale']:.6f})   ceiling drift {drift:+.2f} pp")
        print(f"  vs saved: max {max(d_items)} item(s) differ at any α, max |ΔAUROC| {d_auc:.2e}")

        out = {k: s[k] for k in ("pair_id", "display_label", "source_model", "target_model",
                                 "source_layer", "target_layer", "variant", "ceiling", "trained_sweep")}
        out.update({"procrustes_sweep": rows, "procrustes_scale": proc.scale,
                    "ceiling_drift_pp": drift, "n_train": n_tr, "train_seed": TRAIN_SEED,
                    "max_len": MAX_LEN, "timestamp": datetime.now(timezone.utc).isoformat()})
        storage.save_payload(OUT_DIR, pid, out, fork_path=FORK, filename="procrustes_sweep.json")
        recomputed[pid] = out

        del src_all, tgt_all
        gc.collect(); torch.cuda.empty_cache()

    for pid, r in recomputed.items():
        tr_so, _ = metrics.strict_overshoot(r["trained_sweep"])
        pr_so, _ = metrics.strict_overshoot(r["procrustes_sweep"])
        print(f"\nstrict overshoot  trained {tr_so:+.2f} pp   procrustes {pr_so:+.2f} pp   {pid}")

README.md:   0%|          | 0.00/658 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 65.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.19MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]



Llama-3.2-3B-Instruct L12:   0%|          | 0/3599 [00:00<?, ?it/s]

Llama-3.2-3B-Instruct L12:   0%|          | 9/3599 [00:00<00:40, 87.73it/s]

Llama-3.2-3B-Instruct L12:   1%|          | 18/3599 [00:00<00:41, 85.97it/s]

Llama-3.2-3B-Instruct L12:   1%|          | 27/3599 [00:00<00:40, 87.32it/s]

Llama-3.2-3B-Instruct L12:   1%|          | 36/3599 [00:00<00:40, 87.74it/s]

Llama-3.2-3B-Instruct L12:   1%|▏         | 45/3599 [00:00<00:40, 87.42it/s]

Llama-3.2-3B-Instruct L12:   2%|▏         | 54/3599 [00:00<00:40, 87.49it/s]

Llama-3.2-3B-Instruct L12:   2%|▏         | 63/3599 [00:00<00:40, 87.19it/s]

Llama-3.2-3B-Instruct L12:   2%|▏         | 72/3599 [00:00<00:40, 86.14it/s]

Llama-3.2-3B-Instruct L12:   2%|▏         | 81/3599 [00:00<00:41, 85.61it/s]

Llama-3.2-3B-Instruct L12:   3%|▎         | 91/3599 [00:01<00:39, 87.77it/s]

Llama-3.2-3B-Instruct L12:   3%|▎         | 100/3599 [00:01<00:40, 86.91it/s]

Llama-3.2-3B-Instruct L12:   3%|▎         | 109/3599 [00:01<00:40, 86.7

  scale 0.215412 (saved 0.215412)   ceiling drift +0.00 pp
  vs saved: max 0 item(s) differ at any α, max |ΔAUROC| 1.56e-06

strict overshoot  trained +7.00 pp   procrustes +9.76 pp   qwen2_5-1_5b-instruct_l17_to_llama3_2-3b-instruct_l12__truth


## Part 3 — cross-pair analysis (CPU)

Rebuilds the published table from the saved per-pair files, using the same definitions:
- **peak Δ**: the best α's accuracy minus the target's native ceiling.
- **α=1 Δ**: accuracy at full injection minus the ceiling.
- **Win or loss**: a difference larger than `TIE_PP` in either direction; anything smaller is a tie.

It then checks every row and aggregate against `procrustes_n21_summary.json`.

**Camera-ready numbers.** The published summary counts one run twice
(`llama3_2-3b-instruct_l12_to_qwen2_5-1_5b-instruct_l12` appears as both `__transfer_rsa` and
`__transfer_truth`: same adapter, same probes, identical sweeps). The analysis detects it and
recomputes every aggregate with it counted once. Those 21-pair aggregates are what the camera-ready
quotes, and they are checked against the values pinned in `CAMERA_READY` below.

In [5]:
from scipy.stats import spearmanr

def build_row(r):
    ceil = r["ceiling"]["accuracy"]
    tr_peak = max(r["trained_sweep"], key=lambda x: x["accuracy"])
    pr_peak = max(r["procrustes_sweep"], key=lambda x: x["accuracy"])
    return {
        "name": r["display_label"], "pair_id": r["pair_id"], "ceiling": ceil * 100,
        "tr_dpeak": (tr_peak["accuracy"] - ceil) * 100,
        "pr_dpeak": (pr_peak["accuracy"] - ceil) * 100,
        "tr_da1": (r["trained_sweep"][-1]["accuracy"] - ceil) * 100,
        "pr_da1": (r["procrustes_sweep"][-1]["accuracy"] - ceil) * 100,
        "tr_peak_alpha": tr_peak["alpha"], "pr_peak_alpha": pr_peak["alpha"],
        "tr_peak_auc": tr_peak["auroc"], "pr_peak_auc": pr_peak["auroc"],
        "drift": r.get("ceiling_drift_pp", 0.0),
    }

def aggregate(rows):
    tp = np.array([r["tr_dpeak"] for r in rows]); pp = np.array([r["pr_dpeak"] for r in rows])
    t1 = np.array([r["tr_da1"] for r in rows]);   p1 = np.array([r["pr_da1"] for r in rows])
    def wlt(a, b):
        w, l = int((b > a + TIE_PP).sum()), int((b < a - TIE_PP).sum())
        return w, l, len(rows) - w - l
    wp, lp, tp_ = wlt(tp, pp); w1, l1, t1_ = wlt(t1, p1)
    rp, pp_val = spearmanr(tp, pp); r1, p1_val = spearmanr(t1, p1)
    return {
        "trained_peak_med": float(np.median(tp)), "procrustes_peak_med": float(np.median(pp)),
        "trained_a1_med": float(np.median(t1)),   "procrustes_a1_med": float(np.median(p1)),
        "proc_wins_peak": wp, "proc_loses_peak": lp, "ties_peak": tp_,
        "proc_wins_a1": w1,   "proc_loses_a1": l1,   "ties_a1": t1_,
        "spearman_peak": float(rp), "spearman_peak_p": float(pp_val),
        "spearman_a1": float(r1),   "spearman_a1_p": float(p1_val),
    }

rows = sorted((build_row(saved[p]) for p in POPULATION), key=lambda x: -x["tr_dpeak"])
agg = aggregate(rows)

print(f"{'pair':<46} {'ceil':>6} {'tr pk Δ':>8} {'pr pk Δ':>8} {'tr α1 Δ':>8} {'pr α1 Δ':>8}")
for r in rows:
    print(f"{r['name']:<46} {r['ceiling']:>5.2f}% {r['tr_dpeak']:>+8.2f} {r['pr_dpeak']:>+8.2f} "
          f"{r['tr_da1']:>+8.2f} {r['pr_da1']:>+8.2f}{'  ⚠ drift' if abs(r['drift']) > 1 else ''}")

n = len(rows)
print(f"\npeak  : Procrustes wins {agg['proc_wins_peak']}, ties {agg['ties_peak']}, loses {agg['proc_loses_peak']}"
      f"  → matches or beats trained on {agg['proc_wins_peak'] + agg['ties_peak']}/{n}")
print(f"α = 1 : Procrustes wins {agg['proc_wins_a1']}, ties {agg['ties_a1']}, loses {agg['proc_loses_a1']}"
      f"  → matches or beats trained on {agg['proc_wins_a1'] + agg['ties_a1']}/{n}")
print(f"medians  peak Δ: trained {agg['trained_peak_med']:+.2f}  procrustes {agg['procrustes_peak_med']:+.2f}   "
      f"α=1 Δ: trained {agg['trained_a1_med']:+.2f}  procrustes {agg['procrustes_a1_med']:+.2f}")
print(f"Spearman  peak ρ={agg['spearman_peak']:+.3f} (p={agg['spearman_peak_p']:.3f})   "
      f"α=1 ρ={agg['spearman_a1']:+.3f} (p={agg['spearman_a1_p']:.3f})")

# Two directory names can hold the same run (same models, layers, and sweeps)
def _sig(r):
    return (r["source_model"], r["source_layer"], r["target_model"], r["target_layer"],
            tuple(x["accuracy"] for x in r["trained_sweep"]),
            tuple(x["accuracy"] for x in r["procrustes_sweep"]))
seen, duplicates = {}, {}
for p in POPULATION:
    k = _sig(saved[p])
    if k in seen: duplicates[p] = seen[k]
    else: seen[k] = p
for d, c in duplicates.items():
    print(f"\n⚠ duplicate run in population: {d}\n                       same as: {c}")

# Camera-ready population: every run counted once
uniq = [r for r in rows if r["pair_id"] not in duplicates]
agg_unique = aggregate(uniq)
n_u = len(uniq)
print(f"\n--- counted once (n={n_u}) — the numbers the camera-ready quotes ---")
print(f"peak  : Procrustes wins {agg_unique['proc_wins_peak']}, ties {agg_unique['ties_peak']}, "
      f"loses {agg_unique['proc_loses_peak']}  → matches or beats trained on "
      f"{agg_unique['proc_wins_peak'] + agg_unique['ties_peak']}/{n_u}")
print(f"α = 1 : Procrustes wins {agg_unique['proc_wins_a1']}, ties {agg_unique['ties_a1']}, "
      f"loses {agg_unique['proc_loses_a1']}  → matches or beats trained on "
      f"{agg_unique['proc_wins_a1'] + agg_unique['ties_a1']}/{n_u}")
print(f"medians  peak Δ: trained {agg_unique['trained_peak_med']:+.2f}  procrustes {agg_unique['procrustes_peak_med']:+.2f}   "
      f"α=1 Δ: trained {agg_unique['trained_a1_med']:+.2f}  procrustes {agg_unique['procrustes_a1_med']:+.2f}")
print(f"Spearman  peak ρ={agg_unique['spearman_peak']:+.3f} (p={agg_unique['spearman_peak_p']:.3f})   "
      f"α=1 ρ={agg_unique['spearman_a1']:+.3f} (p={agg_unique['spearman_a1_p']:.3f})")

pair                                             ceil  tr pk Δ  pr pk Δ  tr α1 Δ  pr α1 Δ
Qwen2.5-1.5B (L17) → Llama-3.2-3B (L12)        77.05%   +12.51   +17.01   -15.38   -10.69
Qwen2.5-7B (L21) → Mistral-7B (L13)            82.80%   +11.38   +12.63   -32.40   -23.39
Llama-3.2-3B (L12) → Mistral-7B (L13)          82.80%   +10.57    +0.00   -30.08    -5.13
Llama-3-8B (L12) → Mistral-7B (L13)            82.80%   +10.44   +11.63   -32.27    +0.13
Llama-3.2-3B (L12) → Mistral-7B (L14)          82.80%   +10.19    +0.00   -25.20    -6.50
Llama-3-8B (L14) → Mistral-7B (L14)            82.80%    +9.13   +10.63   -32.40    +6.44
Llama-3.2-3B (L12) → Qwen2.5-1.5B (L12)        58.47%    +8.51   +19.39    -8.94   +10.38
Llama-3.2-3B (L12) → Qwen2.5-1.5B (L12)        58.47%    +8.51   +19.39    -8.94   +10.38
Qwen2.5-1.5B (L12) → Llama-3.2-3B (L12)        77.05%    +7.63    +8.19   -28.08   -23.45
Qwen2.5-1.5B (L12) → Gemma-2-2B (L11)          81.74%    +7.00   +11.19   -30.64   -25.33
Llama-3.2-

In [6]:
# --- regression check against the published summary ------------------------
FIELDS = ("ceiling", "tr_dpeak", "pr_dpeak", "tr_da1", "pr_da1", "tr_peak_alpha",
          "pr_peak_alpha", "tr_peak_auc", "pr_peak_auc", "drift")
ref_rows = {r["pair_id"]: r for r in reference["rows"]}
new_rows = {r["pair_id"]: r for r in rows}
bad = [p for p in POPULATION
       if any(abs(float(ref_rows[p][k]) - float(new_rows[p][k])) > 1e-9 for k in FIELDS)]
bad_agg = [k for k, v in reference["aggregate"].items() if abs(float(v) - float(agg[k])) > 1e-9]

if reference["n_pairs"] != len(rows):
    raise SystemExit(f"population size {len(rows)} ≠ published n_pairs {reference['n_pairs']}")
if bad or bad_agg:
    raise SystemExit(f"REGRESSION — rows: {bad}  aggregates: {bad_agg}")
print(f"✓ reproduces {REFERENCE_SUMMARY} exactly ({len(rows)} rows, {len(agg)} aggregates)")


CAMERA_READY = {
    "n_pairs": 21,
    "proc_wins_peak": 13, "ties_peak": 5, "proc_loses_peak": 3,
    "proc_wins_a1": 20,   "ties_a1": 0,   "proc_loses_a1": 1,
    "trained_peak_med": 0.6253908692933074,
    "procrustes_peak_med": 4.5653533458411495,
    "trained_a1_med": -30.64415259537211,
    "procrustes_a1_med": -10.7567229518449,
    "spearman_peak": 0.3648730727251417,
    "spearman_peak_p": 0.1038834316395333,
    "spearman_a1": 0.4320988566491288,
    "spearman_a1_p": 0.0504479972390995,
}
got_cr = {"n_pairs": len(uniq), **agg_unique}
bad_cr = [k for k, v in CAMERA_READY.items() if abs(float(v) - float(got_cr[k])) > 1e-9]
if bad_cr:
    raise SystemExit(f"CAMERA-READY REGRESSION — {[(k, CAMERA_READY[k], got_cr[k]) for k in bad_cr]}")
print(f"✓ camera-ready aggregates reproduce ({len(uniq)} unique pairs, {len(CAMERA_READY) - 1} values)")

✓ reproduces procrustes_n21_summary.json exactly (22 rows, 14 aggregates)
✓ camera-ready aggregates reproduce (21 unique pairs, 14 values)


In [ ]:
# --- save -------------------------------------------------------------------
out = {
    "n_pairs": len(rows), "pair_ids": POPULATION, "tie_pp": TIE_PP,
    "rows": rows, "aggregate": agg, "duplicates": duplicates,
    "n_pairs_unique": len(uniq), "pair_ids_unique": [r["pair_id"] for r in uniq],
    "aggregate_unique": agg_unique,
    "parity": {"max_len": MAX_LEN, "truncation_noop_on_eval": bool(parity_ok),
               "pairs_with_zero_ceiling_drift": len(POPULATION) - len(nonzero)},
    "recomputed": sorted(recomputed),
    "provenance": storage.provenance(),
}
out_path = Path("/content/procrustes_analysis.json")
json.dump(out, open(out_path, "w"), indent=2)
print(f"saved -> {out_path}")

saved -> /content/procrustes_analysis.json
